In [13]:
import os
try:
    import google.colab
    DI_COLAB = True
except ImportError:
    DI_COLAB = False

def pastikan_file(nama):
    if not os.path.exists(nama):
        if not DI_COLAB:
            raise FileNotFoundError(f"{nama} tidak ditemukan di {os.getcwd()}")
        from google.colab import files
        print(f"Unggah file: {nama}")
        up = files.upload()
        for k in up:
            if k != nama and nama.split(".")[0] in k:   # Colab kadang menambah akhiran (1)
                os.rename(k, nama)
    assert os.path.exists(nama), f"{nama} masih belum ada"

def unduh(*nama_file):
    if DI_COLAB:
        from google.colab import files
        for f in nama_file:
            files.download(f)

In [7]:
import warnings; warnings.filterwarnings("ignore")
import json, itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (11, 4)

## 1.1 Membaca file sumber
File Excel berisi satu sheet `uraian` dengan blok kolom per tahun (Target, SSPD, Realisasi, %).

In [8]:
NAMA_FILE = "Data Pendukung BPHTB PAK 2026 dan Target 2027.xlsx"
pastikan_file(NAMA_FILE)
raw = pd.read_excel(NAMA_FILE, sheet_name="uraian", header=None)
print("Ukuran sheet:", raw.shape)
raw.iloc[:18, :6]

Ukuran sheet: (27, 48)


,0,1,2,3,4,5
0,NaN,ESTIMASI TARGET DAN REALISASI BPHTB 2026,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,Bulan,TARGET 2021,Realisasi 2021,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,JAN- DES,SSPD,Realisasi,%
5,NaN,Januari,32039190112,470,22440938424,0.700422
6,NaN,Februari,51116283736,1181,75584553575.5,1.478679
7,NaN,Maret,73006125233,1501,89811470210,1.230191
8,NaN,April,77013969368,1585,102760125400,1.334305
9,NaN,Mei,67678942779,1154,78781858264,1.164053


## 1.2 Memetakan posisi kolom
Baris 5–16 berisi Januari–Desember, baris 17 berisi Total. Kolom realisasi tiap tahun:

| Tahun | Kolom target | Kolom realisasi | Keterangan |
|---|---|---|---|
| 2021–2024 | 2, 6, 10, 14 | 4, 8, 12, 16 | Realisasi |
| 2025 | 18 | 24 | Total Realisasi (Jual Beli + Non Jual Beli) |
| 2026 | 26 | 32 | Total Realisasi; Jan–Agu riil, Sep–Des estimasi |

In [9]:
BARIS_BULAN = list(range(5, 17))
BARIS_TOTAL = 17
KOL_REALISASI = {2021: 4, 2022: 8, 2023: 12, 2024: 16, 2025: 24, 2026: 32}
KOL_TARGET    = {2021: 2, 2022: 6, 2023: 10, 2024: 14, 2025: 18, 2026: 26}

# cek label kolom agar pemetaan tidak salah
def judul_terdekat(baris, k):
    """Ambil label terisi terdekat di kiri kolom k (judul blok tahun memakai sel gabungan)."""
    for j in range(k, -1, -1):
        if pd.notna(raw.iloc[baris, j]):
            return raw.iloc[baris, j]
for th, k in KOL_REALISASI.items():
    print(th, "| blok:", judul_terdekat(2, k), "| kolom:", raw.iloc[4, k], "| Januari:", f"{raw.iloc[5, k]:,.0f}")

2021 | blok: Realisasi 2021 | kolom: Realisasi | Januari: 22,440,938,424
2022 | blok: Realisasi 2022 | kolom: Realisasi | Januari: 39,324,989,826
2023 | blok: Realisasi 2023 | kolom: Realisasi | Januari: 44,268,974,548
2024 | blok: Realisasi 2024 | kolom: Realisasi | Januari: 60,131,277,079
2025 | blok: Tahun 2025 | kolom: Total Realisasi | Januari: 44,197,462,537
2026 | blok: Tahun 2026 | kolom: Total Realisasi | Januari: 51,426,133,424


## 1.3 Ekstraksi ke deret waktu bulanan

In [10]:
nilai = []
for th, k in KOL_REALISASI.items():
    v = raw.loc[BARIS_BULAN, k].astype(float).values
    nilai.extend(v[:8] if th == 2026 else v)          # 2026: hanya Jan–Agu yang riil

y = pd.Series(np.array(nilai) / 1e9,
              index=pd.date_range("2021-01-01", periods=len(nilai), freq="MS"),
              name="realisasi_miliar")
y.index.name = "bulan"

estimasi_bapenda = pd.Series(raw.loc[BARIS_BULAN[8:], 32].astype(float).values / 1e9,
                             index=pd.date_range("2026-09-01", periods=4, freq="MS"),
                             name="estimasi_bapenda_miliar")
estimasi_bapenda.index.name = "bulan"

target = pd.Series({th: float(raw.loc[BARIS_TOTAL, k]) / 1e9 for th, k in KOL_TARGET.items()},
                   name="target_miliar")
target.index.name = "tahun"
y.head(14)

,realisasi_miliar
bulan,
2021-01-01,22.440938
2021-02-01,75.584554
2021-03-01,89.811470
2021-04-01,102.760125
2021-05-01,78.781858
2021-06-01,97.660501
2021-07-01,100.526332
2021-08-01,77.079336
2021-09-01,99.199513


## 1.4 Validasi data

In [11]:
realisasi_th = y[:"2025"].groupby(y[:"2025"].index.year).sum()
total_excel = pd.Series({th: float(raw.loc[BARIS_TOTAL, KOL_REALISASI[th]]) / 1e9 for th in range(2021, 2026)})
validasi = pd.DataFrame({"Total hasil ekstraksi": realisasi_th.values, "Total di Excel": total_excel.values},
                        index=range(2021, 2026))
validasi["Selisih"] = validasi.iloc[:, 0] - validasi.iloc[:, 1]
print(validasi.round(3))
assert np.allclose(validasi["Selisih"], 0, atol=1e-6), "Ekstraksi tidak cocok dengan Excel!"

print("\nNilai hilang:", y.isna().sum(), "| Nilai <= 0:", (y <= 0).sum())
print("Frekuensi lengkap tanpa bulan bolong:", len(y) == len(pd.date_range(y.index[0], y.index[-1], freq="MS")))

      Total hasil ekstraksi  Total di Excel  Selisih
2021               1291.169        1291.169      0.0
2022               1280.817        1280.817      0.0
2023               1414.189        1414.189      0.0
2024               1342.012        1342.012      0.0
2025               1418.268        1418.268      0.0

Nilai hilang: 0 | Nilai <= 0: 0
Frekuensi lengkap tanpa bulan bolong: True


## 1.5 Menyimpan dataset bersih

In [12]:
y.to_csv("bphtb_bulanan.csv")
estimasi_bapenda.to_csv("estimasi_bapenda_sep_des_2026.csv")
target.to_csv("target_bphtb.csv")
print(pd.concat([target.rename("Target (M)"),
                 y.groupby(y.index.year).sum().rename("Realisasi (M)")], axis=1).round(1))
unduh("bphtb_bulanan.csv", "estimasi_bapenda_sep_des_2026.csv", "target_bphtb.csv")

      Target (M)  Realisasi (M)
2021      1320.1         1291.2
2022      1565.5         1280.8
2023      1442.5         1414.2
2024      1469.8         1342.0
2025      1645.1         1418.3
2026      1556.5          794.3


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>